# Lecture: Visualizing Text Data

Text contains information, but we need to turn it into features before we can summarize and visualize it. We will compare legitimate SMS messages with spam and identify words that might help distinguish the two groups.


## Learning goals

- Prepare text reproducibly while preserving the original messages.
- Compare word clouds, word-frequency bar charts, and a word-frequency scatter plot.
- Distinguish exact counts and percentages of messages.
- Compare text-derived quantitative features across groups.
- Identify vocabulary that could help distinguish spam from legitimate messages.
- Explain why word frequencies alone do not establish how well a detector would work.


## The reproducible analysis workflow

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can help answer it?
3. **Evidence:** What should we calculate and display?
4. **Check:** Does the evidence make sense in context and agree with the original records?
5. **Conclusion:** What claim is supported?
6. **Limitation:** What should we avoid concluding?


## Install additional libraries if needed

Run this cell if `wordcloud` or `scikit-learn` is not installed in your environment. The text-preparation and word-cloud code is provided so we can focus on interpreting the evidence.


In [ ]:
%pip install wordcloud scikit-learn


## Import the libraries


In [ ]:
import html
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


## Step 1: Question

**Which words could help distinguish SMS spam from legitimate messages?**

If we were to build a spam detector, what words would we look for? We will compare how often words appear in the two groups and how many messages contain them.

### Discussion

Which words would you expect to appear more often in spam? Could legitimate messages contain those same words?


## Step 2: Data

The course CSV is a screened version of the [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection), shared under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The original collection contains 5,574 labeled messages from several sources. The course labels are `legitimate` and `spam`.

Classroom screening removed 221 whole messages containing stronger language, explicit or suggestive content, or a few sensitive references. The remaining 5,353 messages retain their original wording and categories. Occasional mild language may remain. Screening affects the sample; this is a historical collection, not a sample of current incoming texts.

| Feature | Description |
| --- | --- |
| `message_id` | Original record number; an identifier, not a measurement |
| `label` | Known category: `legitimate` or `spam` |
| `message` | Original SMS text |

Load the data and inspect its records, dimensions, storage types, and missing values.


In [ ]:
sms_df = pd.read_csv('data/sms_screened.csv')
sms_df.head()


In [ ]:
sms_df.tail()


In [ ]:
print(sms_df.shape)


In [ ]:
sms_df.info()


In [ ]:
print(sms_df.isna().sum())


### Discussion: check the data

What does one row represent? Complete the table from the inspection output.

| Feature | Feature type | Data storage type | Missing values |
| --- | --- | --- | --- |
| `message_id` | | | |
| `label` | | | |
| `message` | | | |


## Step 3: Evidence

### How many messages are in each group?

Use all 5,353 screened messages in this analysis. The bar chart shows exact counts. The table also reports each category as a percentage of **all messages in the dataset**. Repeated texts remain because each row is a record in the collection.


In [ ]:
label_order = ['legitimate', 'spam']
label_titles = {'legitimate': 'Legitimate messages', 'spam': 'Spam'}
message_counts = sms_df['label'].value_counts()
print(message_counts)
print(100 * sms_df['label'].value_counts(normalize=True))

sns.countplot(data=sms_df, x='label', order=label_order, color='teal')
plt.xlabel('Message category')
plt.ylabel('Number of messages')
plt.title('Number of legitimate and spam messages')
plt.show()


### Prepare the text

We will estimate message length using whitespace-separated words, convert text to lowercase, remove URLs and placeholders, and extract words. A **regular expression** describes a text pattern: the pattern below extracts sequences of letters and apostrophes. `explode()` places each extracted word in its own row.

**Stopwords** are common words that we choose to leave out of the frequency displays. We also omit tokens shorter than three characters. These choices simplify the displays but discard information, including short SMS abbreviations. Keep the original `message` so we can revisit its context.

We start with scikit-learn's English stopword list. Add lowercase words to `additional_stopwords` to extend it; leave the list empty to use the default. For example, try `["i'm", "dont"]`, then rerun the preparation and figures. Apply the same list to both categories.

### Discussion

Which common SMS words would you add, and why? Could removing a word also remove a useful clue about spam?


In [ ]:
sms_df['length_words'] = sms_df['message'].str.split().str.len()
sms_df['prepared_text'] = (
    sms_df['message'].map(html.unescape).str.lower()
    .str.replace(r'https?://\S+|www\.\S+', ' ', regex=True)
    .str.replace(r'<[^>]*>', ' ', regex=True)
)
sms_df['words'] = sms_df['prepared_text'].str.findall(r"\b[a-z']+\b")
words_df = sms_df[['message_id', 'label', 'words']].explode('words')
words_df = words_df.rename(columns={'words': 'word'})

additional_stopwords = []  # For example: ["i'm", "dont"]
stopwords = set(ENGLISH_STOP_WORDS).union(additional_stopwords)

words_df = words_df.loc[
    ~words_df['word'].isin(stopwords) & words_df['word'].str.len().ge(3)
].copy()


### Compare word clouds

A **word cloud** uses larger type for more frequent words. Each cloud scales its own words, so font sizes should be compared within a cloud, not between clouds. Placement and color are decorative; nearby words are not necessarily related.

The provided code sets the image dimensions, uses a white background, displays up to 75 individual words, and fixes the random seed so the arrangement is reproducible.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for axis, label in zip(axes, label_order):
    counts = words_df.loc[words_df['label'].eq(label), 'word'].value_counts()
    cloud = WordCloud(width=850, height=500, background_color='white',
                      max_words=75, collocations=False, random_state=412)
    cloud.generate_from_frequencies(counts)
    axis.imshow(cloud, interpolation='bilinear')
    axis.set_title(label_titles[label])
    axis.axis('off')
plt.tight_layout()
plt.show()


### Discussion

What words stand out in each group? Which might help identify spam? What comparison is difficult to make using the clouds?


### Compare word occurrences

The grouped bar chart counts every occurrence of a word, including repetitions in the same message. We display the eight most frequent remaining words from each category, combining the two lists so both groups contribute words. The same selection will appear in the next figure.

There are more legitimate messages than spam messages, so raw counts give the larger group more opportunities to contain a word.


In [ ]:
frequency_df = words_df.groupby(['label', 'word']).agg(
    count=('word', 'size'), messages_with_word=('message_id', 'nunique')
).reset_index()
frequency_df['percent_messages'] = (
    100 * frequency_df['messages_with_word'] / frequency_df['label'].map(message_counts)
)

selected_words = []
for label in label_order:
    top = frequency_df.loc[frequency_df['label'].eq(label)].nlargest(8, 'count')['word']
    selected_words.extend(word for word in top if word not in selected_words)

# Include zero values when a selected word does not appear in one category.
comparison = frequency_df.set_index(['label', 'word']).reindex(
    pd.MultiIndex.from_product([label_order, selected_words], names=['label', 'word']),
    fill_value=0
).reset_index()

plt.figure(figsize=(9, 7))
sns.barplot(data=comparison, x='count', y='word', hue='label',
            order=selected_words, hue_order=label_order,
            palette={'legitimate': 'teal', 'spam': 'darkorange'})
plt.xlabel('Number of word occurrences')
plt.ylabel('Word')
plt.title('Word occurrences in legitimate and spam messages')
plt.legend(title='Message category')
plt.tight_layout()
plt.show()


### What percentage of each message category contains a word?

For each word, divide the number of **legitimate messages containing it by all 4,676 legitimate messages**, and do the same using **all 677 spam messages** for spam. Each message counts once for a word, even if it repeats that word.

For example, 58 of 4,676 legitimate messages contain “free” (1.24%), compared with 158 of 677 spam messages (23.34%). These percentages describe how common “free” is within each group. They do not describe how the messages containing “free” are divided between legitimate and spam.


In [ ]:
plt.figure(figsize=(9, 7))
sns.barplot(data=comparison, x='percent_messages', y='word', hue='label',
            order=selected_words, hue_order=label_order,
            palette={'legitimate': 'teal', 'spam': 'darkorange'})
plt.xlabel('Percent of messages in each category containing the word')
plt.ylabel('Word')
plt.title('How common is each word within legitimate messages and within spam?')
plt.legend(title='Denominator: all messages in this category')
plt.tight_layout()
plt.show()


### Discussion

1. Which comparisons change when you use percentages instead of raw counts? Why?
2. Which words are more common in spam as a percentage of messages? Which are more common in legitimate messages?
3. If a word occurs three times in one message, how does that affect each measure?
4. Why do the percentages of messages containing the displayed words not have to add to 100%?


### Compare word frequencies with a scatter plot

Each point represents one word. Its horizontal position shows the percentage of **legitimate messages** containing it; its vertical position shows the percentage of **spam messages** containing it. Both axes use the same scale.

The dashed diagonal marks equal percentages. Words above the line are more common within spam; words below it are more common within legitimate messages. Distance from the line describes a difference in percentages, not statistical significance or detector accuracy.

To keep the figure readable, display words appearing in at least 1% of messages in either category. This includes more words than the bar charts but omits rare words. Hover to identify a word and read both percentages; zoom to separate nearby points.


In [ ]:
word_comparison = frequency_df.pivot(
    index='word', columns='label', values='percent_messages'
).fillna(0).reset_index()

common_words = word_comparison.loc[
    (word_comparison['legitimate'] >= 1) | (word_comparison['spam'] >= 1)
]
axis_max = common_words[['legitimate', 'spam']].max().max() * 1.05

fig = px.scatter(
    common_words, x='legitimate', y='spam', hover_name='word',
    hover_data={'legitimate': ':.2f', 'spam': ':.2f'},
    labels={'legitimate': 'Percent of legitimate messages containing the word',
            'spam': 'Percent of spam messages containing the word'},
    range_x=[0, axis_max], range_y=[0, axis_max],
    title='Which words are more common in spam?'
)
fig.add_shape(type='line', x0=0, y0=0, x1=axis_max, y1=axis_max,
              line={'color': 'gray', 'dash': 'dash'})
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(width=750, height=750)
fig.show()


### Discussion

1. Hover over a word well above the line and one below it. What do their positions tell you?
2. Find a word close to the line. Would its presence alone help distinguish the categories?
3. Why does a word on the vertical axis have a legitimate-message percentage of zero?
4. What does this figure make easier to compare than the bar charts? What information is omitted by the 1% display threshold?


### Compare message lengths

Message length is a quantitative feature constructed from text. Use a box plot and group summaries to compare its distribution across labels. The Check section provides a function for reading original messages in context.


In [ ]:
sns.boxplot(data=sms_df, x='label', y='length_words', order=label_order, color='teal')
plt.xlabel('Message category')
plt.ylabel('Approximate message length (words)')
plt.title('SMS message lengths')
plt.show()
print(sms_df.groupby('label')['length_words'].describe().round(1))


### Discussion

Which group has the higher median? How much do the distributions overlap? Would a long message always be spam?


## Step 4: Check

Return to original messages to check what a word means in context. The function below prints up to three randomly selected messages from each category containing the selected word. Change `word` to explore another word or `n` to print more examples.

It uses the individual words extracted during preparation, before stopword removal. Matching ignores capitalization and matches a complete word, so “free” does not match “freedom.” The fixed random seed keeps the examples reproducible. If a category has fewer than the requested number of matches, it prints all available matches; if there are none, it reports that.


In [ ]:
def show_message_examples(data, word, n=3):
    word = word.strip().lower()
    contains_word = data['words'].apply(lambda words: word in words)

    for category in ['legitimate', 'spam']:
        matches = data.loc[contains_word & data['label'].eq(category)]
        print(f'\n{category.capitalize()}: {len(matches)} messages containing "{word}"')
        if matches.empty:
            print('No matching messages.')
        else:
            examples = matches.sample(n=min(n, len(matches)), random_state=42)
            for row in examples.itertuples():
                print(f'\nMessage {row.message_id}: {row.message}')


### Illustration 1: “free”

Read examples from both categories. How does the meaning or purpose of “free” differ between the messages?


In [ ]:
show_message_examples(sms_df, 'free')


### Illustration 2: “claim”

Compare these examples with “free.” Are there matches in both categories? What should we avoid concluding if a word appears in only one category in this dataset?


In [ ]:
show_message_examples(sms_df, 'claim')


### Discussion

What do these examples reveal that the frequency charts cannot? Use the bar charts to compare how widespread each word is within legitimate messages and within spam. Why do a few examples alone not tell us how common a word is?


## Step 5: Conclusion

**Which words could help distinguish SMS spam from legitimate messages?**

Words such as “free,” “claim,” and “mobile” are promising clues because they appear in a larger percentage of spam messages. Word clouds give an initial impression, while bar charts make exact frequencies and relative frequencies easier to compare. The scatter plot highlights words that are more common within one category by comparing their positions with the equal-percentage line. A word can help distinguish groups without appearing in every spam message or being absent from all legitimate messages.

### Discussion

Choose two words you would investigate for a possible spam detector. Support each choice with the percentage of messages containing it in each category. Explain why its raw count alone would be less informative.


## Step 6: Limitation

Word counts lose order and context. Stopword choices, token lengths, spelling, and abbreviations affect the displays. The same word can have different meanings in different messages.

This historical collection combines different sources, and classroom screening removed a larger share of spam than legitimate messages. Repeated messages remain and can make repeated vocabulary more prominent. Its language patterns and spam frequency may differ from a real inbox today.

We have compared vocabulary rather than built or evaluated a detector. Before claiming that these words identify new spam reliably, we would need to examine how a detector using them performs on new, independently labeled messages.

### Discussion

What information does word frequency leave out? How might that affect the words you would look for in a spam detector?


## What happened to the original text?

| Representation | What it preserves | What it loses |
| --- | --- | --- |
| Original message | Wording, order, and context | Difficult to summarize across thousands of messages |
| Message length | Approximate amount of text | Nearly all meaning |
| Word frequencies | Repeated vocabulary | Word order and message context |

Quantification makes comparison possible, but each conversion discards information. Return to original messages when a pattern needs context.
